# Vérification de la limite communale et du découpage (clipping)

Ce notebook vérifie que `download.limites_administratives.fetch_commune_boundary` récupère la vraie limite officielle de la commune (ADMIN EXPRESS IGN, filtrée par code INSEE) et que `processing.clipping.clip_to_boundary` découpe correctement une couche sur cette limite, avec un buffer optionnel pour conserver une légère continuité des données autour de la commune.

Test réalisé sur la couche `batiment` (BD TOPO) déjà téléchargée.

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

import geopandas as gpd

from core.config import load_entities
from download.limites_administratives import fetch_commune_boundary
from processing.clipping import clip_to_boundary

## 1. Entité testée (issue du CSV)

In [ ]:
csv_path = repo_root / "data" / "configuration" / "entites_exemple.csv"
entity = load_entities(csv_path)[0]
entity

## 2. Limite communale officielle (ADMIN EXPRESS IGN)

In [ ]:
boundary = fetch_commune_boundary(entity.code_insee)
boundary[["nom_officiel", "code_insee", "population", "superficie_cadastrale", "geometry"]]

## 3. Découpage de la couche `batiment` (avec et sans buffer)

In [ ]:
batiment_path = repo_root / "data" / "raw" / "vector" / "bd_topo" / "batiment.gpkg"
processed_path = repo_root / "data" / "processed" / "vector" / "bd_topo" / "batiment.gpkg"

original = gpd.read_file(batiment_path)
clipped_strict = clip_to_boundary(batiment_path, boundary, buffer_distance=0)
clipped_buffer = clip_to_boundary(batiment_path, boundary, buffer_distance=50, output_path=processed_path)

print("Batiments dans le fichier original (bbox) :", len(original))
print("Batiments apres decoupage strict (buffer=0) :", len(clipped_strict))
print("Batiments apres decoupage avec buffer 50m :", len(clipped_buffer))
print("Couche traitee ecrite dans :", processed_path)

## 4. Visualisation avant / après

Bâtiments hors commune (gris), limite communale (rouge), bâtiments conservés avec buffer 50m (orange), bâtiments conservés en découpage strict (vert). Reprojection en EPSG:4326 uniquement pour l'affichage. Backend `folium`, plus fiable pour le rendu des widgets dans VS Code.

In [ ]:
import leafmap.foliumap as leafmap

m = leafmap.Map()
m.add_gdf(
    original.to_crs(epsg=4326),
    layer_name="Batiments (bbox, avant decoupage)",
    style={"color": "gray", "fillOpacity": 0.2},
)
m.add_gdf(
    clipped_buffer.to_crs(epsg=4326),
    layer_name="Batiments conserves (buffer 50m)",
    style={"color": "orange", "fillOpacity": 0.5},
)
m.add_gdf(
    clipped_strict.to_crs(epsg=4326),
    layer_name="Batiments conserves (decoupage strict)",
    style={"color": "green", "fillOpacity": 0.5},
)
m.add_gdf(
    boundary.to_crs(epsg=4326),
    layer_name="Limite communale",
    style={"color": "red", "fillOpacity": 0},
)
m.zoom_to_gdf(boundary.to_crs(epsg=4326))
m